In [ ]:
"""
Genera shots_data.json y copia match_predictions.json al dashboard/data/
para que el dashboard pueda leerlos con fetch().

Ejecutar desde la raíz del proyecto:
  python3 notebooks/generar_datos_dashboard.py
"""
import pandas as pd
import numpy as np
import json
import pickle
import requests
from pathlib import Path

In [ ]:
ROOT = Path(__file__).resolve().parent.parent
DATA_DIR     = ROOT / "data"
MODELS_DIR   = ROOT / "models"
DASH_DATA    = ROOT / "dashboard" / "data"
DASH_DATA.mkdir(exist_ok=True)

In [ ]:
API_BASE = "https://premier.72-60-245-2.sslip.io"

In [ ]:
# ── 1. Cargar eventos (tiros) ────────────────────────────────────────────
print("⬇️  Cargando eventos/tiros desde el API ...")
ruta_events = DATA_DIR / "shots_raw.csv"
if ruta_events.exists():
    print("   Usando caché local (shots_raw).")
    events = pd.read_csv(ruta_events)
else:
    # Solo tiros (más rápido que los 444K eventos completos)
    resp = requests.get(f"{API_BASE}/events?is_shot=true&limit=10000", timeout=60)
    resp.raise_for_status()
    shots_raw = resp.json().get("events", [])
    events = pd.DataFrame(shots_raw)
    events.to_csv(DATA_DIR / "shots_raw.csv", index=False)
    print(f"   {len(events)} tiros descargados.")

In [ ]:
# Filtrar solo tiros
if 'is_shot' in events.columns:
    shots = events[events['is_shot'] == True].copy()
else:
    shots = events.copy()

In [ ]:
print(f"   Total tiros: {len(shots)}")

In [ ]:
# ── 2. Cargar modelo xG y scaler ────────────────────────────────────────
print("📦 Cargando modelo xG ...")
with open(MODELS_DIR / "model_xg.pkl", "rb") as f:
    model_xg = pickle.load(f)
with open(MODELS_DIR / "scaler_xg.pkl", "rb") as f:
    scaler_xg = pickle.load(f)
with open(MODELS_DIR / "features_xg.pkl", "rb") as f:
    FEATURES_XG = pickle.load(f)

In [ ]:
# ── 3. Feature engineering sobre tiros ──────────────────────────────────
print("🔧 Generando features de tiros ...")

In [ ]:
q = shots['qualifiers'].fillna('[]').astype(str)

In [ ]:
shots['x_norm'] = np.where(shots['x'] < 50, 100 - shots['x'], shots['x'])
shots['y_norm'] = np.where(shots['x'] < 50, 100 - shots['y'], shots['y'])

In [ ]:
shots['distance'] = np.sqrt((100 - shots['x_norm'])**2 + (50 - shots['y_norm'])**2)
shots['angle']    = np.degrees(np.arctan2(50 - shots['y_norm'], 100 - shots['x_norm']))

In [ ]:
# Eliminar atípicos para que no aparezcan en el dashboard
shots = shots[(~q.str.contains('OwnGoal', na=False)) & (shots['distance'] <= 50)].copy()
q = shots['qualifiers'].fillna('[]').astype(str)

In [ ]:
shots['is_big_chance'] = q.str.contains('BigChance',    na=False).astype(int)
shots['is_penalty']    = q.str.contains('"Penalty"',    na=False).astype(int)
shots['is_header']     = q.str.contains('"Head"',       na=False).astype(int)
shots['is_right_foot'] = q.str.contains('RightFoot',    na=False).astype(int)
shots['is_left_foot']  = q.str.contains('LeftFoot',     na=False).astype(int)
shots['is_counter']    = q.str.contains('FastBreak',    na=False).astype(int)
shots['is_assisted']   = q.str.contains('IntentionalAssist|KeyPass', na=False).astype(int)
shots['is_volley']     = q.str.contains('Volley',       na=False).astype(int)
shots['zone_box_centre']  = q.str.contains('BoxCentre',      na=False).astype(int)
shots['zone_out_of_box']  = q.str.contains('OutOfBox',        na=False).astype(int)
shots['zone_small_box']   = q.str.contains('SmallBox',        na=False).astype(int)

In [ ]:
# Predecir xG con el modelo
available = [f for f in FEATURES_XG if f in shots.columns]
X_shots = shots[available].fillna(0)
X_shots_sc = scaler_xg.transform(X_shots)
xg_probs = model_xg.predict_proba(X_shots_sc)[:, 1]
shots['xg_prob'] = xg_probs

In [ ]:
# ── 4. Construir shots_data.json ────────────────────────────────────────
print("💾 Construyendo shots_data.json ...")
cols_out = ['x', 'y', 'is_goal', 'xg_prob', 'is_big_chance',
            'is_penalty', 'is_header', 'distance', 'team_name']

In [ ]:
# Asegurar columna is_goal
if 'is_goal' not in shots.columns and 'isGoal' in shots.columns:
    shots['is_goal'] = shots['isGoal']

In [ ]:
output = []
for _, row in shots.iterrows():
    output.append({
        'x':           round(float(row.get('x_norm', 0)), 2),
        'y':           round(float(row.get('y_norm', 50)), 2),
        'is_goal':     bool(row.get('is_goal', False)),
        'xg_prob':     round(float(row['xg_prob']), 4),
        'is_big_chance': int(row.get('is_big_chance', 0)),
        'is_penalty':    int(row.get('is_penalty', 0)),
        'is_header':     int(row.get('is_header', 0)),
        'distance':      round(float(row.get('distance', 0)), 2),
        'team_name':     str(row.get('team_name', 'Unknown'))
    })

In [ ]:
(DASH_DATA / 'shots_data.json').write_text(json.dumps(output, ensure_ascii=False))
print(f"   ✅ shots_data.json: {len(output)} tiros → {DASH_DATA/'shots_data.json'}")

In [ ]:
# ── 5. Copiar match_predictions.json ────────────────────────────────────
import shutil
src = DATA_DIR / "match_predictions.json"
dst = DASH_DATA / "match_predictions.json"
match_preds = []
if src.exists():
    shutil.copy(src, dst)
    print(f"   ✅ match_predictions.json copiado → {dst}")
    with open(src, 'r') as f:
        match_preds = json.load(f)
else:
    print("   ⚠️  match_predictions.json no encontrado en data/ — ejecuta primero modelo2_match_predictor.py")

In [ ]:
# ── 6. Generar data.js (Para evitar errores CORS sin servidor) ────────────
js_content = f"window.SHOTS_DATA = {json.dumps(output, ensure_ascii=False)};\n"
js_content += f"window.MATCH_PREDICTIONS = {json.dumps(match_preds, ensure_ascii=False)};\n"
(DASH_DATA / 'data.js').write_text(js_content)
print(f"   ✅ data.js generado para soporte offline → {DASH_DATA/'data.js'}")

In [ ]:
print("\n✅ Datos del dashboard listos en dashboard/data/")
print("   - shots_data.json")
print("   - match_predictions.json")
print("   - data.js")
print("\n🚀 Ya puedes abrir el archivo index.html directamente en tu navegador (doble clic) sin problemas.")